# 23.6 工作流编排:Airflow / Prefect / Dagster / Workflow Orchestration

**中文**:真实的数据/ML 工作从来不是"一个脚本跑到底",而是**一串有依赖关系的步骤**:抽取数据 → 清洗转换 → 训练模型 → 评估 → 部署,每步依赖上一步,还要**定时运行、失败自动重试、某步挂了通知你、能回填历史数据、能监控整条链路**。手动 `python a.py && python b.py` 一旦复杂就崩溃。**工作流编排器(orchestrator)** 就是干这个的:把你的流水线定义成一张 **DAG(有向无环图)**,由调度器负责按依赖顺序执行、重试、监控、告警。**Airflow** 是事实标准,**Prefect / Dagster** 是更现代的后起之秀。本节从零实现一个 DAG 编排器,亲手实现它的所有核心能力——**拓扑排序执行、失败重试、失败传播(上游挂了跳过下游)**,让你彻底理解 Airflow 底层在做什么。
**English**: Real data/ML work is never "one script running end to end" but **a chain of dependent steps**: extract data → clean/transform → train model → evaluate → deploy, each depending on the previous, plus needing to **run on a schedule, auto-retry on failure, notify you when a step fails, backfill historical data, and monitor the whole chain**. Manual `python a.py && python b.py` collapses once things get complex. A **workflow orchestrator** does exactly this: define your pipeline as a **DAG (directed acyclic graph)**, and the scheduler executes it in dependency order with retries, monitoring, and alerting. **Airflow** is the de-facto standard; **Prefect / Dagster** are more modern successors. This section builds a DAG orchestrator from scratch, implementing all its core capabilities — **topological execution, failure retries, failure propagation (skip downstream when upstream fails)** — so you fully understand what Airflow does underneath.

---

**中文**:**编排器的核心概念**:
**English**: **The orchestrator's core concepts**:
- **中文**:**DAG(有向无环图)**:整个流水线就是一张 DAG——节点是**任务(task)**,边是**依赖(dependency)**。无环保证不会死循环。`train` 依赖 `transform`,`transform` 依赖 `extract`。
  **DAG (directed acyclic graph)**: the whole pipeline is a DAG — nodes are **tasks**, edges are **dependencies**. Acyclicity guarantees no infinite loops. `train` depends on `transform`, `transform` on `extract`.
- **中文**:**拓扑执行(topological execution)**:调度器按依赖顺序运行——一个任务只有在它**所有上游都成功后**才开始;没有依赖关系的任务可以**并行**。
  **Topological execution**: the scheduler runs in dependency order — a task starts only after **all its upstreams succeed**; independent tasks can run **in parallel**.
- **中文**:**重试(retry)**:任务可能因**暂时性故障**失败(网络抖动、资源临时不够)。编排器自动重试 N 次,给它自愈的机会。
  **Retry**: tasks may fail from **transient issues** (network jitter, temporary resource shortage). The orchestrator auto-retries N times, giving it a chance to self-heal.
- **中文**:**失败传播(failure propagation)**:如果一个任务最终失败,它的**所有下游任务被跳过**(既然训练失败了,就不该部署)——而不是傻乎乎地继续。
  **Failure propagation**: if a task ultimately fails, all its **downstream tasks are skipped** (if training failed, don't deploy) — rather than blindly continuing.
- **中文**:**调度 + 回填(schedule + backfill)**:按 cron 定时触发(每天凌晨跑);**回填**能对"过去某段时间"补跑(如新上线的管道要处理历史数据)。**幂等性(idempotency)** 是前提:同一个任务重跑结果一致,才能安全重试和回填。
  **Schedule + backfill**: cron-triggered on a schedule (run nightly); **backfill** re-runs for "a past time range" (e.g. a newly deployed pipeline processing historical data). **Idempotency** is a prerequisite: a task rerun gives the same result, enabling safe retries and backfills.

> 💡 **面试速查 / Interview cheat-sheet（★★ 数据工程必考）**
> **中文**:**工作流编排**=把流水线定义成 **DAG**(任务+依赖), 调度器负责拓扑执行、重试、监控、告警、回填。**核心概念**:DAG(有向无环)、task、依赖、拓扑执行(上游全成功才跑, 无依赖可并行)、**重试**(应对暂时性故障)、**失败传播**(上游失败→跳过下游)、调度(cron)、**回填 backfill**、**幂等性**(重跑结果一致, 是重试/回填的前提)、sensor(等待外部条件)、XCom(任务间传数据)。**三大工具**:**Airflow**(事实标准, 成熟生态, Python 定义 DAG, 但较重/调度偏静态)、**Prefect**(更 Pythonic/动态, 少样板)、**Dagster**(**资产导向**——以数据资产而非任务为中心, 强类型/可测试/数据感知)。**关键原则**:任务要**幂等 + 原子**(失败可安全重试)、避免任务间隐式状态、监控 SLA。**vs cron**:cron 只定时不管依赖/重试/回填/监控; 编排器全包。**云托管**:AWS MWAA、GCP Composer(都是托管 Airflow)、Astronomer。面试金句:*"数据/ML 管道是有依赖的 DAG, 用编排器(Airflow/Prefect/Dagster)按依赖拓扑执行、失败重试、上游失败传播跳过下游、定时调度和回填; 前提是任务幂等原子(才能安全重试); Airflow 是标准但偏重, Dagster 以数据资产为中心更现代; 比裸 cron 多了依赖管理/重试/监控/回填。"*
> **English**: **Workflow orchestration** = define the pipeline as a **DAG** (tasks + dependencies); the scheduler handles topological execution, retries, monitoring, alerting, backfill. **Core concepts**: DAG (acyclic), task, dependency, topological execution (run only after all upstreams succeed, parallel where independent), **retry** (for transient failures), **failure propagation** (upstream fails → skip downstream), scheduling (cron), **backfill**, **idempotency** (rerun gives the same result, prerequisite for retry/backfill), sensor (wait for external conditions), XCom (pass data between tasks). **Three tools**: **Airflow** (de-facto standard, mature ecosystem, Python-defined DAGs, but heavier / more static scheduling), **Prefect** (more Pythonic/dynamic, less boilerplate), **Dagster** (**asset-oriented** — centered on data assets not tasks, strongly-typed/testable/data-aware). **Key principle**: tasks should be **idempotent + atomic** (safe to retry on failure), avoid implicit state between tasks, monitor SLAs. **vs cron**: cron only schedules, ignoring dependencies/retries/backfill/monitoring; orchestrators handle all. **Cloud-managed**: AWS MWAA, GCP Composer (both managed Airflow), Astronomer. Interview line: *"Data/ML pipelines are dependent DAGs; use an orchestrator (Airflow/Prefect/Dagster) for dependency-topological execution, failure retries, upstream-failure propagation skipping downstream, scheduling, and backfill; the prerequisite is idempotent atomic tasks (safe to retry); Airflow is the standard but heavier, Dagster is more modern being data-asset-centered; versus bare cron it adds dependency management/retries/monitoring/backfill."*


In [ ]:

# ============================================================
# 从零实现 DAG 编排器 / DAG orchestrator from scratch
# 中文:实现 Airflow 的核心:任务+依赖构成 DAG, 拓扑排序执行, 失败自动重试, 上游失败则跳过下游。
# English: implement Airflow's core: tasks + dependencies form a DAG, topological execution, auto-retry on failure,
#      skip downstream when upstream fails.
# ============================================================
class Task:
    def __init__(self, name, fn, retries=0):
        self.name=name; self.fn=fn; self.retries=retries; self.deps=[]
    def after(self, *upstreams): self.deps.extend(upstreams); return self   # 声明依赖 / declare dependencies
class DAG:
    def __init__(self): self.tasks={}
    def add(self, task): self.tasks[task.name]=task; return task
    def _toposort(self):                                       # 拓扑排序:上游排在下游前面 / topological sort
        order=[]; seen=set()
        def visit(t):
            if t.name in seen: return
            for d in t.deps: visit(d)                          # 先访问所有依赖 / visit dependencies first
            seen.add(t.name); order.append(t)
        for t in self.tasks.values(): visit(t)
        return order
    def run(self):
        status={}
        for t in self._toposort():                            # 按拓扑序执行 / execute in topological order
            if any(status.get(d.name)!="success" for d in t.deps):        # 上游有失败/跳过 → 跳过本任务 / failure propagation
                status[t.name]="skipped"; print(f"  ⏭  {t.name:12} SKIPPED (上游失败, 失败传播)"); continue
            for attempt in range(t.retries+1):                # 自动重试 / auto-retry
                try:
                    t.fn(); status[t.name]="success"
                    print(f"  ✅ {t.name:12} SUCCESS" + (f"  (重试 {attempt} 次后成功)" if attempt else "")); break
                except Exception as e:
                    if attempt<t.retries: print(f"  🔁 {t.name:12} 失败, 重试 ({attempt+1}/{t.retries})…")
                    else: status[t.name]="failed"; print(f"  ❌ {t.name:12} FAILED: {e}")
        return status

# 一条典型的数据/ML 管道 DAG:extract →(transform_a, transform_b 并行)→ train → deploy
dag=DAG()
extract = dag.add(Task("extract",     lambda: None))
tr_a    = dag.add(Task("transform_a", lambda: None).after(extract))
_n=[0]
def flaky_transform():                                        # 一个"暂时性故障"任务:前两次失败, 第三次成功 / transient-failure task
    _n[0]+=1
    if _n[0]<3: raise RuntimeError("暂时性网络错误 transient network error")
tr_b    = dag.add(Task("transform_b", flaky_transform, retries=3).after(extract))   # 靠重试自愈 / self-heal via retries
train   = dag.add(Task("train",       lambda: None).after(tr_a, tr_b))
deploy  = dag.add(Task("deploy",      lambda: None).after(train))
print("运行 #1:transform_b 前两次失败, 靠重试成功 → 全流程完成 / retries save a flaky task:")
dag.run()


In [ ]:

# ============================================================
# 失败传播:硬失败的任务会跳过所有下游 / failure propagation: a hard failure skips all downstream
# 中文:train 硬失败(无重试)→ deploy 被跳过。编排器绝不会在训练失败后还去部署——这正是它比裸脚本聪明的地方。
# English: train hard-fails (no retries) → deploy is skipped. The orchestrator never deploys after training fails —
#      exactly where it's smarter than a bare script.
# ============================================================
dag2=DAG()
e =dag2.add(Task("extract",     lambda: None))
ta=dag2.add(Task("transform_a", lambda: None).after(e))
tb=dag2.add(Task("transform_b", lambda: None).after(e))
def training_diverges(): raise ValueError("模型训练发散 / OOM(硬失败)")
tr=dag2.add(Task("train",  training_diverges).after(ta, tb))   # 硬失败, 无重试 / hard failure, no retries
dp=dag2.add(Task("deploy", lambda: None).after(tr))            # 依赖 train → 会被跳过 / depends on train → skipped
print("运行 #2:train 硬失败 → 下游 deploy 自动跳过(不会部署一个训练失败的模型):")
final=dag2.run()
print("\n各任务最终状态 / final status:", final)
print("→ 编排器保证:训练失败就绝不部署。裸脚本 `train.py && deploy.py` 靠 shell 的 && 也能挡, 但没有重试/监控/回填/告警")


In [ ]:

# ============================================================
# 可视化:DAG 结构 + 编排器 vs cron / DAG structure + orchestrator vs cron
# ============================================================
import matplotlib.pyplot as plt
fig,ax=plt.subplots(1,2,figsize=(14,5))
# ① DAG 图 / the DAG
ax[0].axis("off"); ax[0].set_title("数据/ML 管道 DAG(任务+依赖)",fontsize=12,weight="bold")
pos={"extract":(0.1,0.5),"transform_a":(0.4,0.72),"transform_b":(0.4,0.28),"train":(0.7,0.5),"deploy":(0.93,0.5)}
edges=[("extract","transform_a"),("extract","transform_b"),("transform_a","train"),("transform_b","train"),("train","deploy")]
for a,b in edges:
    ax[0].annotate("",xy=pos[b],xytext=pos[a],arrowprops=dict(arrowstyle="->",color="gray",lw=1.5),transform=ax[0].transAxes)
for n,(x,y) in pos.items():
    ax[0].add_patch(plt.Circle((x,y),0.07,fc="#4C72B0",alpha=0.7,transform=ax[0].transAxes))
    ax[0].text(x,y,n.replace("transform_","t_"),ha="center",va="center",fontsize=7,color="white",transform=ax[0].transAxes)
ax[0].text(0.5,0.03,"拓扑执行:上游全成功才跑下游; t_a/t_b 无依赖可并行",ha="center",fontsize=8,style="italic",transform=ax[0].transAxes)
# ② 编排器 vs cron 能力对比 / orchestrator vs cron
ax[1].axis("off"); ax[1].set_title("编排器 vs 裸 cron",fontsize=12,weight="bold")
caps=["依赖管理(DAG)","失败自动重试","失败传播(跳过下游)","调度 + 回填 backfill","监控 + 告警","并行执行","可视化 UI"]
for i,c in enumerate(caps):
    y=0.85-i*0.12
    ax[1].text(0.05,y,c,fontsize=9,transform=ax[1].transAxes)
    ax[1].text(0.62,y,"✅",fontsize=11,transform=ax[1].transAxes)     # orchestrator
    ax[1].text(0.82,y,"❌" if i<5 else "❌",fontsize=11,transform=ax[1].transAxes)  # cron
ax[1].text(0.62,0.97,"编排器",fontsize=9,weight="bold",transform=ax[1].transAxes)
ax[1].text(0.82,0.97,"cron",fontsize=9,weight="bold",transform=ax[1].transAxes)
plt.tight_layout(); plt.savefig("/tmp/cloud06_viz.png",dpi=80); plt.show()
print("左:管道是 DAG, 按依赖拓扑执行; 右:编排器比裸 cron 多了依赖/重试/失败传播/回填/监控/并行/UI")


**中文**:诚实解读:
**English**: Honest takeaways:

**中文**:
1. **编排器把"一堆脚本"升级为"可靠的生产管道"**:一个数据/ML 管道有五个步骤听起来简单,但生产中的魔鬼在细节:第 3 步偶尔因网络抖动失败怎么办?第 4 步失败了还该不该跑第 5 步?每天凌晨自动跑怎么保证?上周的数据要补跑怎么办?哪一步慢了、挂了怎么第一时间知道?我们几十行的 DAG 编排器就实现了这些核心能力:**拓扑执行**(按依赖顺序、无依赖并行)、**重试**(让 transform_b 从暂时性故障中自愈)、**失败传播**(train 挂了就绝不部署)。这些不是"高级功能",而是任何生产管道的**基本生存能力**——没有它们,你的管道会在半夜静默地部署一个训练失败的模型,或者因为一次网络抖动整条链路白跑。
2. **幂等性是编排的隐形基石,也是最容易忽视的设计原则**:重试和回填之所以能安全进行,前提是任务**幂等(idempotent)**——同一个任务重跑一次,结果和跑一次完全一样,不会重复扣款、不会写入重复数据、不会累加错误。如果你的"写入"任务不幂等(比如每次运行都往表里 append),那么一次重试就会造成数据重复。**正确的做法是让任务原子且幂等**:用"覆盖分区"而非"追加"、用 upsert(有则更新无则插入)而非 insert、用确定性的输出路径(接 21.11 湖仓的 MERGE、22.9 特征平台)。**"任务必须幂等"是编排设计的第一原则**,理解它比会写 Airflow DAG 语法重要得多。
3. **诚实的工具选择与边界**:①**Airflow vs Prefect vs Dagster**:**Airflow** 是事实标准(生态最成熟、招聘最认、云托管齐全 MWAA/Composer),但它较重、调度模型偏静态(为定时批处理设计)、本地开发体验一般;**Prefect** 更 Pythonic、动态、少样板,适合更灵活的工作流;**Dagster** 提出了**资产导向(asset-oriented)** 的新范式——不以"任务"为中心,而以"数据资产"为中心(这张表由什么生成、被谁消费),强类型、可测试、数据感知,是数据工程理念的一次进化。选型看团队偏好和成熟度要求;新项目 Dagster/Prefect 值得考虑,但 Airflow 仍是最安全的求职技能。②**别把编排器当计算引擎**:编排器负责"**协调**"(什么时候、按什么顺序、跑什么),不负责"**计算**"(真正的数据处理应该在 Spark/DuckDB/SQL 里做,编排器只是触发它们)。一个常见反模式是把大量数据处理逻辑塞进 Airflow 任务本身,让调度节点不堪重负——正确的是让任务只做"提交一个 Spark 作业 / 触发一个 SQL",重活交给专门的引擎。③**vs 裸 cron**:cron 能定时,但不管依赖、不重试、不传播失败、不回填、不监控——小到两三步的简单定时任务 cron 够用,复杂管道必须上编排器。**结论:工作流编排器把有依赖的数据/ML 管道定义成 DAG, 提供拓扑执行、重试、失败传播、调度、回填、监控——这是把脚本升级为可靠生产管道的关键; 核心设计原则是任务幂等原子(才能安全重试回填); Airflow 是标准、Dagster 以数据资产为中心更现代; 记住编排器只负责协调、不负责计算(重活交给 Spark/SQL 引擎)。**

**English**:
1. **The orchestrator upgrades "a bunch of scripts" into "a reliable production pipeline"**: a five-step data/ML pipeline sounds simple, but the devil in production is in the details: what if step 3 occasionally fails from network jitter? if step 4 fails, should step 5 still run? how to guarantee it runs nightly? how to backfill last week's data? how to know immediately when a step is slow or failed? Our few-dozen-line DAG orchestrator implements these core capabilities: **topological execution** (dependency order, parallel where independent), **retries** (letting transform_b self-heal from a transient failure), **failure propagation** (train fails → never deploy). These aren't "advanced features" but the **basic survival skills** of any production pipeline — without them, your pipeline silently deploys a training-failed model at midnight, or a single network jitter wastes the whole chain.
2. **Idempotency is orchestration's invisible cornerstone, and the most-overlooked design principle**: retries and backfills can proceed safely only if tasks are **idempotent** — rerunning the same task gives exactly the same result as running it once, with no double charges, no duplicate data written, no accumulated errors. If your "write" task isn't idempotent (e.g. appends to a table each run), one retry causes data duplication. **The right approach is atomic and idempotent tasks**: "overwrite partition" instead of "append," upsert (update if exists, insert if not) instead of insert, deterministic output paths (per 21.11's lakehouse MERGE, 22.9's feature store). **"Tasks must be idempotent" is orchestration's first design principle**, understanding it matters far more than knowing Airflow DAG syntax.
3. **Honest tool choice and limits**: ① **Airflow vs Prefect vs Dagster**: **Airflow** is the de-facto standard (most mature ecosystem, most recognized in hiring, full cloud-managed MWAA/Composer), but heavier, with a static-leaning scheduling model (designed for scheduled batch) and mediocre local dev; **Prefect** is more Pythonic, dynamic, less boilerplate, for more flexible workflows; **Dagster** proposes a new **asset-oriented** paradigm — centered not on "tasks" but on "data assets" (what generates this table, who consumes it), strongly-typed, testable, data-aware, an evolution of data-engineering philosophy. Choose by team preference and maturity needs; for new projects Dagster/Prefect are worth considering, but Airflow remains the safest job-seeking skill. ② **Don't treat the orchestrator as a compute engine**: the orchestrator handles "**coordination**" (when, in what order, run what), not "**computation**" (actual data processing should happen in Spark/DuckDB/SQL, the orchestrator just triggers them). A common anti-pattern is stuffing heavy data-processing logic into Airflow tasks themselves, overwhelming the scheduler nodes — the right way is tasks that only "submit a Spark job / trigger a SQL," leaving heavy work to dedicated engines. ③ **vs bare cron**: cron schedules but ignores dependencies, retries, failure propagation, backfill, monitoring — for a simple two-or-three-step scheduled job cron suffices, but complex pipelines need an orchestrator. **Conclusion: a workflow orchestrator defines dependent data/ML pipelines as a DAG, providing topological execution, retries, failure propagation, scheduling, backfill, and monitoring — key to upgrading scripts into reliable production pipelines; the core design principle is idempotent atomic tasks (safe to retry/backfill); Airflow is the standard, Dagster is more modern being data-asset-centered; remember the orchestrator only coordinates, not computes (leave heavy work to Spark/SQL engines).**

> 💼 **实战视角 / Practical angle**
> **中文**:编排落地:①**用 DAG 定义管道**(Airflow/Prefect/Dagster), 声明任务依赖, 让调度器管拓扑执行/重试/失败传播;②**任务必须幂等原子**——覆盖分区而非追加、upsert 而非 insert、确定性输出路径(才能安全重试/回填);③**编排只协调不计算**——任务里只提交 Spark 作业/触发 SQL/调 API, 重活交专门引擎;④**配好重试策略**(暂时性故障)、**告警**(失败/SLA 超时通知 Slack/PagerDuty)、**回填**(新管道补历史);⑤**云托管** AWS MWAA / GCP Composer / Astronomer(托管 Airflow), Databricks Workflows/Prefect Cloud;⑥新项目考虑 Dagster(数据资产导向、可测试)或 Prefect(轻量动态), 但 Airflow 是最稳的求职技能。面试金句:*"数据/ML 管道是有依赖的 DAG, 编排器(Airflow/Prefect/Dagster)负责拓扑执行、失败重试、上游失败传播跳过下游、定时调度、回填、监控告警; 核心原则是任务幂等原子才能安全重试回填; 编排器只协调不计算, 重活交 Spark/SQL; Airflow 是标准, Dagster 以数据资产为中心更现代。"*
> **English**: Orchestration in practice: ① **define pipelines as DAGs** (Airflow/Prefect/Dagster), declare task dependencies, let the scheduler handle topological execution/retries/failure propagation; ② **tasks must be idempotent and atomic** — overwrite partitions not append, upsert not insert, deterministic output paths (safe to retry/backfill); ③ **orchestration coordinates, doesn't compute** — tasks only submit Spark jobs/trigger SQL/call APIs, leaving heavy work to dedicated engines; ④ **configure retry policies** (transient failures), **alerting** (notify Slack/PagerDuty on failure/SLA breach), **backfill** (new pipelines processing history); ⑤ **cloud-managed** AWS MWAA / GCP Composer / Astronomer (managed Airflow), Databricks Workflows/Prefect Cloud; ⑥ for new projects consider Dagster (data-asset-oriented, testable) or Prefect (lightweight, dynamic), but Airflow is the most solid job-seeking skill. Interview line: *"Data/ML pipelines are dependent DAGs; an orchestrator (Airflow/Prefect/Dagster) handles topological execution, failure retries, upstream-failure propagation skipping downstream, scheduling, backfill, monitoring/alerting; the core principle is idempotent atomic tasks for safe retry/backfill; the orchestrator only coordinates, not computes, leaving heavy work to Spark/SQL; Airflow is the standard, Dagster is more modern being data-asset-centered."*

---
### 小结 / Summary
- **中文**:编排器把数据/ML 管道定义成 DAG(任务+依赖), 提供拓扑执行、重试、失败传播、调度、回填、监控——脚本→可靠生产管道。
- **English**: An orchestrator defines data/ML pipelines as a DAG (tasks + dependencies), providing topological execution, retries, failure propagation, scheduling, backfill, monitoring — scripts → reliable production pipelines.
- **中文**:核心原则:任务幂等原子(才能安全重试/回填); 编排只协调不计算(重活交 Spark/SQL 引擎)。
- **English**: Core principle: idempotent atomic tasks (safe to retry/backfill); orchestration coordinates, not computes (heavy work to Spark/SQL engines).
- **中文**:Airflow 是事实标准, Prefect 轻量动态, Dagster 以数据资产为中心更现代; 比裸 cron 多依赖/重试/监控/回填。
- **English**: Airflow is the de-facto standard, Prefect is lightweight/dynamic, Dagster is more modern being data-asset-centered; more than bare cron (dependencies/retries/monitoring/backfill).
